# AQI predictor — exploratory analysis

EDA on the CPCB `city_day.csv` data. Model training and the published metrics live in `train.py` (run `python train.py`); this notebook only explores the data and loads those results.

Run from the repo root's `notebooks/` folder.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from aqi import FEATURES
from aqi.categories import CATEGORIES

df = pd.read_csv(ROOT / 'data' / 'city_day.csv', parse_dates=['Date'])
df.shape

## Coverage and missing values

In [ ]:
print(df.Date.min().date(), 'to', df.Date.max().date(), '|', df.City.nunique(), 'cities')
df[FEATURES + ['AQI']].isna().mean().sort_values(ascending=False).round(3)

In [ ]:
df.groupby(df.Date.dt.year)['AQI'].count().plot.bar(title='Rows with a known AQI, by year');

## Target distribution

In [ ]:
df['AQI_Bucket'].value_counts().reindex(CATEGORIES).plot.bar(title='AQI category counts');

In [ ]:
df['AQI'].describe()

## Relationships

AQI is defined from pollutant sub-indices, so strong correlation with PM2.5 / PM10 is expected.

In [ ]:
df[FEATURES + ['AQI']].corr()['AQI'].drop('AQI').sort_values().plot.barh(title='Correlation with AQI');

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column='PM2.5', by='AQI_Bucket', ax=ax, showfliers=False, positions=None)
ax.set_title('PM2.5 by AQI category'); fig.suptitle('');

## Model results

Chronological split: trained on 2015–2018, tested on 2019–2020. Model chosen on a validation window (Jul–Dec 2018) before the test set was used.

In [ ]:
metrics = json.loads((ROOT / 'model' / 'metrics.json').read_text())
print('Selected:', metrics['selected_model'])
pd.DataFrame(metrics['test']).T